<a href="https://colab.research.google.com/github/batz02/DiffTexture-SR/blob/main/AI_assisted_CG_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Installazione e import

In [ ]:
!pip install -q diffusers transformers opencv-python einops omegaconf pytorch_lightning xformers accelerate torch==2.9.0

from google.colab import drive
drive.mount('/content/drive')

path_dir = '/content/drive/MyDrive/images/'

import torch
import torch.nn as nn
import gc
from einops import rearrange, repeat

#### Maschera binaria

In [ ]:
import random
from PIL import Image, ImageOps, ImageChops
import numpy as np
import os
import matplotlib.pyplot as plt

def save_source_augmentations(source_img, output_dir):

    aug_dir = os.path.join(output_dir, "source_augs")
    os.makedirs(aug_dir, exist_ok=True)

    transforms = {
        "source": source_img,
        "rot_90": source_img.rotate(90, expand=False),
        "rot_180": source_img.rotate(180, expand=False),
        "rot_45":  source_img.rotate(45, expand=False, resample=Image.BICUBIC),
        "rot_135": source_img.rotate(135, expand=False, resample=Image.BICUBIC),
        "mirror_y": ImageOps.mirror(source_img)
    }

    for name, img in transforms.items():
        save_path = os.path.join(aug_dir, f"{name}.jpg")
        img.save(save_path)

    print(f"Augmentations salvate in: {aug_dir}")

def prepare_texture_inputs(
    source_texture_path,
    output_dir,
    canvas_size=(512, 512),
    num_fragments=5,
    min_scale=0.5,
    max_scale=1.5,
    rotation_range=(0, 360),
    max_attempts=100
):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    source = Image.open(source_texture_path).convert("RGB")
    if max(source.size) > 1024:
        source.thumbnail((512, 512), Image.LANCZOS)

    save_source_augmentations(source, output_dir)

    w_src, h_src = source.size

    temp_src_for_bg = source.resize(canvas_size, Image.LANCZOS)
    image_array = np.array(temp_src_for_bg)
    height, width, channels = image_array.shape
    flattened_pixels = image_array.reshape(-1, channels)
    np.random.shuffle(flattened_pixels)
    shuffled_image_array = flattened_pixels.reshape(height, width, channels)

    canvas = Image.fromarray(shuffled_image_array.astype(np.uint8))

    mask = Image.new("L", canvas_size, 0)

    ref_dir = os.path.join(output_dir, "refs")
    os.makedirs(ref_dir, exist_ok=True)

    count_placed = 0

    for i in range(num_fragments):
        placed_successfully = False

        crop_size_base = min(canvas_size) // 2
        crop_size = random.randint(crop_size_base - 50, crop_size_base + 50)
        crop_size = min(crop_size, min(w_src, h_src))

        x = random.randint(0, w_src - crop_size)
        y = random.randint(0, h_src - crop_size)
        crop = source.crop((x, y, x + crop_size, y + crop_size))

        scale = random.uniform(min_scale, max_scale)
        new_size = (int(crop_size * scale), int(crop_size * scale))
        crop_resized = crop.resize(new_size, Image.LANCZOS)

        angle = random.randint(0,4) * 45
        crop_rotated = crop_resized.rotate(angle, expand=True, fillcolor=None)

        patch_mask = Image.new("L", new_size, 255)
        patch_mask_rotated = patch_mask.rotate(angle, expand=True, fillcolor=0)

        w_res, h_res = crop_rotated.size

        for attempt in range(max_attempts):
            min_x = -w_res // 4
            max_x = canvas_size[0] - w_res + w_res // 4

            if max_x <= min_x:
                paste_x = (canvas_size[0] - w_res) // 2
            else:
                paste_x = random.randint(min_x, max_x)

            min_y = -h_res // 4
            max_y = canvas_size[1] - h_res + h_res // 4

            if max_y <= min_y:
                paste_y = (canvas_size[1] - h_res) // 2
            else:
                paste_y = random.randint(min_y, max_y)

            temp_check_mask = Image.new("L", canvas_size, 0)
            temp_check_mask.paste(patch_mask_rotated, (paste_x, paste_y), patch_mask_rotated)

            overlap = np.bitwise_and(np.array(mask), np.array(temp_check_mask))

            if np.any(overlap > 0):
                continue
            else:
                canvas.paste(crop_rotated, (paste_x, paste_y), patch_mask_rotated)
                mask.paste(255, (paste_x, paste_y), patch_mask_rotated)

                crop_rotated.save(os.path.join(ref_dir, f"ref_{count_placed}.jpg"))

                placed_successfully = True
                count_placed += 1
                break

        if not placed_successfully:
            print(f"Warning: Impossibile posizionare frammento {i} (spazio insufficiente).")

    target_path = os.path.join(output_dir, "target_collage.jpg")
    mask_path = os.path.join(output_dir, "mask.png")

    canvas.save(target_path)
    mask = ImageOps.invert(mask)
    mask.save(mask_path)

    return target_path, mask_path, ref_dir

INPUT_TEXTURE = f"{path_dir}/refs/10010.jpg"
OUTPUT_PREP_DIR = f"{path_dir}/prepared_input"

tgt_path, mask_path, refs_folder = prepare_texture_inputs(
    INPUT_TEXTURE,
    OUTPUT_PREP_DIR,
    num_fragments=4,
    min_scale=0.3,
    max_scale=0.9,
    rotation_range=(0, 4),
    max_attempts=100
)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(Image.open(tgt_path))
axs[0].set_title("Input Target (Collage)")
axs[1].imshow(Image.open(mask_path), cmap="gray")
axs[1].set_title("Maschera")

aug_path = os.path.join(OUTPUT_PREP_DIR, "source_augs", "rot_45.jpg")
if os.path.exists(aug_path):
    axs[2].imshow(Image.open(aug_path))
    axs[2].set_title("Esempio Augmentation (45°)")
plt.show()

#### Class Attention

In [ ]:
# ------------------------------------------------------------------------
# Module adapted from Official Repository: Self-Rectification / MasaCtrl
# Source: https://github.com/xiaorongjun000/Self-Rectification
# ------------------------------------------------------------------------

class AttentionBase:
    def __init__(self):
        self.cur_step = 0
        self.num_att_layers = -1
        self.cur_att_layer = 0

    def after_step(self):
        pass

    def __call__(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        out = self.forward(q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers:
            self.cur_att_layer = 0
            self.cur_step += 1
            self.after_step()
        return out

    def forward(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        out = torch.einsum('b i j, b j d -> b i d', attn, v)
        out = rearrange(out, '(b h) n d -> b n (h d)', h=num_heads)
        return out

    def reset(self):
        self.cur_step = 0
        self.cur_att_layer = 0


class AttentionStore(AttentionBase):
    def __init__(self, res=[32], min_step=0, max_step=1000):
        super().__init__()
        self.res = res
        self.min_step = min_step
        self.max_step = max_step
        self.valid_steps = 0
        self.self_attns = []
        self.cross_attns = []
        self.self_attns_step = []
        self.cross_attns_step = []

    def after_step(self):
        if self.cur_step > self.min_step and self.cur_step < self.max_step:
            self.valid_steps += 1
            if len(self.self_attns) == 0:
                self.self_attns = self.self_attns_step
                self.cross_attns = self.cross_attns_step
            else:
                for i in range(len(self.self_attns)):
                    self.self_attns[i] += self.self_attns_step[i]
                    self.cross_attns[i] += self.cross_attns_step[i]
        self.self_attns_step.clear()
        self.cross_attns_step.clear()

    def forward(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        if attn.shape[1] <= 64 ** 2:
            if is_cross:
                self.cross_attns_step.append(attn.detach().cpu())
            else:
                self.self_attns_step.append(attn.detach().cpu())
        return super().forward(q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs)


def regiter_attention_editor_diffusers(model, editor: AttentionBase):
    def ca_forward(self, place_in_unet):
        def forward(x, encoder_hidden_states=None, attention_mask=None, context=None, mask=None):
            if encoder_hidden_states is not None:
                context = encoder_hidden_states
            if attention_mask is not None:
                mask = attention_mask

            to_out = self.to_out
            if isinstance(to_out, nn.modules.container.ModuleList):
                to_out = self.to_out[0]
            else:
                to_out = self.to_out

            h = self.heads
            q = self.to_q(x)
            is_cross = context is not None
            context = context if is_cross else x
            k = self.to_k(context)
            v = self.to_v(context)
            q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))

            scale = self.scale if hasattr(self, 'scale') else (q.shape[-1] ** -0.5)
            sim = torch.einsum('b i d, b j d -> b i j', q, k) * scale

            if mask is not None:
                mask = rearrange(mask, 'b ... -> b (...)')
                max_neg_value = -torch.finfo(sim.dtype).max
                mask = repeat(mask, 'b j -> (b h) () j', h=h)
                mask = mask[:, None, :].repeat(h, 1, 1)
                sim.masked_fill_(~mask, max_neg_value)

            attn = sim.softmax(dim=-1)

            out = editor(
                q, k, v, sim, attn, is_cross, place_in_unet,
                self.heads, scale=scale)

            return to_out(out)

        return forward

    def register_editor(net, count, place_in_unet):
        for name, subnet in net.named_children():
            if net.__class__.__name__ == 'Attention':
                net.forward = ca_forward(net, place_in_unet)
                return count + 1
            elif hasattr(net, 'children'):
                count = register_editor(subnet, count, place_in_unet)
        return count

    cross_att_count = 0
    for net_name, net in model.unet.named_children():
        if "down" in net_name:
            cross_att_count += register_editor(net, 0, "down")
        elif "mid" in net_name:
            cross_att_count += register_editor(net, 0, "mid")
        elif "up" in net_name:
            cross_att_count += register_editor(net, 0, "up")
    editor.num_att_layers = cross_att_count
    model.editor = editor


class MutualSelfAttentionControlInversion(AttentionBase):
    MODEL_TYPE = {"SD": 16, "SDXL": 70}

    def __init__(self, start_step=4, start_layer=10, ref_num=1, layer_idx=None, step_idx=None, total_steps=50, model_type="SD"):
        super().__init__()
        self.total_steps = total_steps
        self.total_layers = self.MODEL_TYPE.get(model_type, 16)
        self.start_step = start_step
        self.start_layer = start_layer
        self.layer_idx = layer_idx if layer_idx is not None else list(range(start_layer, self.total_layers))
        self.step_idx = step_idx if step_idx is not None else list(range(0, start_step))
        self.ref_num = ref_num
        print("MasaCtrl at denoising steps: ", self.step_idx)
        print("MasaCtrl at U-Net layers: ", self.layer_idx)

    def attn_batch(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        b = q.shape[0] // num_heads
        q = rearrange(q, "(b h) n d -> h (b n) d", h=num_heads)
        k = rearrange(k, "(b h) n d -> h (b n) d", h=num_heads)
        v = rearrange(v, "(b h) n d -> h (b n) d", h=num_heads)

        sim = torch.einsum("h i d, h j d -> h i j", q, k) * kwargs.get("scale")
        attn = sim.softmax(-1)

        out = torch.einsum("h i j, h j d -> h i d", attn, v)
        out = rearrange(out, "h (b n) d -> b n (h d)", b=b)
        return out

    def forward(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        if is_cross or self.cur_step not in self.step_idx or self.cur_att_layer // 2 not in self.layer_idx:
            return super().forward(q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs)

        out_c_ref = super().forward(q[:num_heads * self.ref_num], k[:num_heads * self.ref_num], v[:num_heads * self.ref_num], sim[:num_heads*self.ref_num][:num_heads*self.ref_num], attn[:num_heads*self.ref_num][:num_heads*self.ref_num], is_cross, place_in_unet, num_heads, **kwargs)
        out_c_tgt = self.attn_batch(q[num_heads * self.ref_num:], k[:num_heads * self.ref_num], v[:num_heads * self.ref_num], sim[:num_heads * self.ref_num], attn, is_cross, place_in_unet, num_heads, **kwargs)
        out = torch.cat([out_c_ref, out_c_tgt], dim=0)
        return out


class MutualSelfAttentionControl(AttentionBase):
    MODEL_TYPE = {"SD": 16, "SDXL": 70}

    def __init__(self, start_step=4, start_layer=10, ref_num=1, layer_idx=None, step_idx=None, total_steps=50, model_type="SD"):
        super().__init__()
        self.total_steps = total_steps
        self.total_layers = self.MODEL_TYPE.get(model_type, 16)
        self.start_step = start_step
        self.start_layer = start_layer
        self.layer_idx = layer_idx if layer_idx is not None else list(range(start_layer, self.total_layers))
        self.step_idx = step_idx if step_idx is not None else list(range(start_step, total_steps))
        self.ref_num = ref_num
        print("MasaCtrl at denoising steps: ", self.step_idx)
        print("MasaCtrl at U-Net layers: ", self.layer_idx)

    def attn_batch(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        b = q.shape[0] // num_heads
        q = rearrange(q, "(b h) n d -> h (b n) d", h=num_heads)
        k = rearrange(k, "(b h) n d -> h (b n) d", h=num_heads)
        v = rearrange(v, "(b h) n d -> h (b n) d", h=num_heads)

        sim = torch.einsum("h i d, h j d -> h i j", q, k) * kwargs.get("scale")
        attn = sim.softmax(-1)

        out = torch.einsum("h i j, h j d -> h i d", attn, v)
        out = rearrange(out, "h (b n) d -> b n (h d)", b=b)
        return out

    def forward(self, q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs):
        if is_cross or self.cur_step not in self.step_idx or self.cur_att_layer // 2 not in self.layer_idx:
            return super().forward(q, k, v, sim, attn, is_cross, place_in_unet, num_heads, **kwargs)

        out_c_ref = super().forward(q[:num_heads * self.ref_num], k[:num_heads * self.ref_num], v[:num_heads * self.ref_num], sim[:num_heads*self.ref_num][:num_heads*self.ref_num], attn[:num_heads*self.ref_num][:num_heads*self.ref_num], is_cross, place_in_unet, num_heads, **kwargs)
        out_c_tgt = self.attn_batch(q[num_heads * self.ref_num:], k[:num_heads * self.ref_num], v[:num_heads * self.ref_num], sim[:num_heads * self.ref_num], attn, is_cross, place_in_unet, num_heads, **kwargs)
        out = torch.cat([out_c_ref, out_c_tgt], dim=0)
        return out

#### Pipeline

In [ ]:
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline
from tqdm import tqdm

# ------------------------------------------------------------------------
# Module adapted from Official Repository: Self-Rectification / MasaCtrl
# Source: https://github.com/xiaorongjun000/Self-Rectification
# ------------------------------------------------------------------------

class MasaCtrlPipeline(StableDiffusionPipeline):

    def next_step(self, model_output, timestep, x, eta=0., verbose=False):
        if verbose: print("timestep: ", timestep)
        next_step = timestep
        timestep = min(timestep - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps, 999)
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep] if timestep >= 0 else self.scheduler.final_alpha_cumprod
        alpha_prod_t_next = self.scheduler.alphas_cumprod[next_step]
        beta_prod_t = 1 - alpha_prod_t
        pred_x0 = (x - beta_prod_t**0.5 * model_output) / alpha_prod_t**0.5
        pred_dir = (1 - alpha_prod_t_next)**0.5 * model_output
        x_next = alpha_prod_t_next**0.5 * pred_x0 + pred_dir
        return x_next, pred_x0

    def step(self, model_output, timestep, x, eta=0.0, verbose=False):
        prev_timestep = timestep - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep > 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_x0 = (x - beta_prod_t**0.5 * model_output) / alpha_prod_t**0.5
        pred_dir = (1 - alpha_prod_t_prev)**0.5 * model_output
        x_prev = alpha_prod_t_prev**0.5 * pred_x0 + pred_dir
        return x_prev, pred_x0

    @torch.inference_mode()
    def image2latent(self, image):
        device = self.vae.device
        dtype = self.vae.dtype

        if type(image) is Image:
            image = np.array(image)
            image = torch.from_numpy(image).float() / 127.5 - 1
            image = image.permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=dtype)
        elif isinstance(image, torch.Tensor):
             image = image.to(device=device, dtype=dtype)

        latents = self.vae.encode(image)['latent_dist'].mean
        latents = latents * 0.18215
        return latents

    @torch.inference_mode()
    def latent2image(self, latents, return_type='np'):
        latents = 1 / 0.18215 * latents.detach()
        image = self.vae.decode(latents)['sample']
        if return_type == 'np':
            image = (image / 2 + 0.5).clamp(0, 1)
            image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
            image = (image * 255).astype(np.uint8)
        elif return_type == "pt":
            image = (image / 2 + 0.5).clamp(0, 1)
        return image

    @torch.inference_mode()
    def __call__(self, prompt, latents, batch_size=1, height=512, width=512, num_inference_steps=50, eta=0.0, unconditioning=None, neg_prompt=None, ref_intermediate_latents=None, return_intermediates=False, **kwds):
        device = self.device
        if isinstance(prompt, list):
            batch_size = len(prompt)
        elif isinstance(prompt, str) and batch_size > 1:
            prompt = [prompt] * batch_size

        text_input = self.tokenizer(prompt, padding="max_length", max_length=77, return_tensors="pt")
        text_embeddings = self.text_encoder(text_input.input_ids.to(device))[0]

        self.scheduler.set_timesteps(num_inference_steps)
        latents_list = [latents]
        pred_x0_list = [latents]

        for i, t in enumerate(tqdm(self.scheduler.timesteps, desc="DDIM Sampler")):
            if ref_intermediate_latents is not None:
                latents_ref = ref_intermediate_latents[-1 - i]
                latents_cur = latents[-1].unsqueeze(0)
                latents = torch.cat([latents_ref, latents_cur])

            model_inputs = latents
            if unconditioning is not None and isinstance(unconditioning, list):

                _, text_embeddings = text_embeddings.chunk(2)
                text_embeddings = torch.cat([unconditioning[i].expand(*text_embeddings.shape), text_embeddings])


            noise_pred = self.unet(model_inputs, t, encoder_hidden_states=text_embeddings).sample
            latents, pred_x0 = self.step(noise_pred, t, latents)

            latents_list.append(latents)
            pred_x0_list.append(pred_x0)

        image = self.latent2image(latents, return_type="pt")

        if not return_intermediates:
            del latents_list, pred_x0_list
            torch.cuda.empty_cache()
            return image

        if return_intermediates:
            pred_x0_list = [self.latent2image(img, return_type="pt") for img in pred_x0_list]
            latents_list = [self.latent2image(img, return_type="pt") for img in latents_list]
            return image, pred_x0_list, latents_list

    @torch.inference_mode()
    def invert(self, image: torch.Tensor, prompt, num_inference_steps=50, guidance_scale=7.5, eta=0.0, return_intermediates=False, ref_intermediate_latents=None, **kwds):
        device = self.device
        batch_size = image.shape[0]
        if isinstance(prompt, list):
            if batch_size == 1:
                image = image.expand(len(prompt), -1, -1, -1)
        elif isinstance(prompt, str) and batch_size > 1:
            prompt = [prompt] * batch_size

        text_input = self.tokenizer(prompt, padding="max_length", max_length=77, return_tensors="pt")
        text_embeddings = self.text_encoder(text_input.input_ids.to(device))[0]

        latents = self.image2latent(image)
        start_latents = latents

        if ref_intermediate_latents is not None:
            text_embeddings = torch.cat([text_embeddings, text_embeddings])

        self.scheduler.set_timesteps(num_inference_steps)
        latents_list = [latents]
        pred_x0_list = [latents]

        latents_cur = latents
        for i, t in enumerate(tqdm(reversed(self.scheduler.timesteps), desc="DDIM Inversion")):
            if ref_intermediate_latents is not None:
                latents_ref = ref_intermediate_latents[-1 - i]
                latents = torch.cat([latents_ref, latents_cur])

            model_inputs = latents
            noise_pred = self.unet(model_inputs, t, encoder_hidden_states=text_embeddings).sample
            latents, pred_x0 = self.next_step(noise_pred, t, latents)

            latents_list.append(latents)
            pred_x0_list.append(pred_x0)
            latents_cur = latents[-1].unsqueeze(0)

        if return_intermediates:
            return latents, latents_list
        return latents, start_latents

#### Main

In [ ]:
import os
import torch
import torch.nn.functional as F
from diffusers import DDIMScheduler
from torchvision.io import read_image
from torchvision.utils import save_image

torch.cuda.empty_cache()
gc.collect()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

def load_image(image_path, res, device, dtype=torch.float16):
    image = read_image(image_path)
    image = image[:3].unsqueeze_(0).float() / 127.5 - 1.
    image = F.interpolate(image, (res, res))
    image = image.to(device, dtype=dtype)
    return image

P1 = 20
P2 = 5
S1 = 20
S2 = 5

out_dir = f"{path_dir}/exp"
os.makedirs(out_dir, exist_ok=True)
sample_count = len(os.listdir(out_dir)) + 1
out_dir = os.path.join(out_dir, f"Sample_{sample_count}")
os.makedirs(out_dir, exist_ok=True)

model_path = "CompVis/stable-diffusion-v1-4"
scheduler = DDIMScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", clip_sample=False, set_alpha_to_one=False)

model = MasaCtrlPipeline.from_pretrained(
    model_path,
    scheduler=scheduler,
    torch_dtype=torch.float16
).to(device)

try:
    model.enable_xformers_memory_efficient_attention()
    print("Xformers enabled!")
except Exception as e:
    print("Xformers not available, trying slicer.", e)
    model.enable_attention_slicing()

model.enable_vae_slicing()


ref_images = torch.cat([
    load_image(f"{path_dir}/prepared_input/source_augs/source.jpg", 512, device),
    load_image(f"{path_dir}/prepared_input/source_augs/rot_90.jpg", 512, device),
    load_image(f"{path_dir}/prepared_input/source_augs/rot_45.jpg", 512, device),
    load_image(f"{path_dir}/prepared_input/source_augs/rot_180.jpg", 512, device),
    load_image(f"{path_dir}/prepared_input/source_augs/rot_135.jpg", 512, device),
    load_image(f"{path_dir}/prepared_input/source_augs/mirror_y.jpg", 512, device)
])

target_image = load_image(f"{path_dir}/prepared_input/target_collage.jpg", 512, device)
mask = load_image(f"{path_dir}/prepared_input/mask.png", 512, device)

ref_num = ref_images.shape[0]

save_image(ref_images.float(), os.path.join(out_dir, f"refs.jpg"), normalize=True)
save_image(target_image.float(), os.path.join(out_dir, f"target.jpg"), normalize=True)
save_image(mask.float(), os.path.join(out_dir, f"mask.jpg"), normalize=True)

print("Running Step 1: Invert References...")
with torch.inference_mode():
    start_code_ref, latents_list_ref = model.invert(
        ref_images,
        [""] * ref_num,
        num_inference_steps=50,
        return_intermediates=True
    )
torch.cuda.empty_cache()


print("Running Step 2: Invert Target (Self)...")
with torch.inference_mode():
    _, latents_list_target_self = model.invert(
        target_image,
        "",
        num_inference_steps=50,
        return_intermediates=True
    )
torch.cuda.empty_cache()

editor = MutualSelfAttentionControlInversion(P1, 10, ref_num=1)
regiter_attention_editor_diffusers(model, editor)

print("Running Step 3: Invert Target (IR)...")
with torch.inference_mode():
    start_code_tgt, _ = model.invert(
        target_image,
        "",
        num_inference_steps=50,
        return_intermediates=True,
        ref_intermediate_latents=latents_list_target_self
    )

start_code = torch.cat([start_code_ref, start_code_tgt])
torch.cuda.empty_cache()

editor = MutualSelfAttentionControl(S1, 10, ref_num=ref_num)
regiter_attention_editor_diffusers(model, editor)

print("Running Step 4: Sampling Result 01...")
with torch.inference_mode():
    image_masactrl = model(
        [""] * (ref_num) + [""],
        latents=start_code,
        ref_intermediate_latents=latents_list_ref
    )
save_image(image_masactrl[-1:].float(), os.path.join(out_dir, f"result_01.jpg"))
torch.cuda.empty_cache()

print("Starting Iteration 2...")
TARGET_PATH = os.path.join(out_dir, f"result_01.jpg")
target_image = load_image(TARGET_PATH, res=512, device=device)

editor = AttentionBase()
regiter_attention_editor_diffusers(model, editor)

editor = MutualSelfAttentionControlInversion(P2, 10, 1)
regiter_attention_editor_diffusers(model, editor)

print("Running Step 5: Invert Result 01...")
with torch.inference_mode():
    start_code_tgt, latents_list_tgt = model.invert(
        target_image,
        "",
        num_inference_steps=50,
        return_intermediates=True,
        ref_intermediate_latents=latents_list_target_self
    )

start_code = torch.cat([start_code_ref, start_code_tgt])
torch.cuda.empty_cache()

editor = MutualSelfAttentionControl(S2, 10, ref_num=ref_num)
regiter_attention_editor_diffusers(model, editor)

print("Running Step 6: Sampling Result 02...")
with torch.inference_mode():
    image_masactrl = model(
        [""] * (ref_num) + [""],
        latents=start_code,
        ref_intermediate_latents=latents_list_ref
    )
save_image(image_masactrl[-1:].float(), os.path.join(out_dir, f"result_02.jpg"))
torch.cuda.empty_cache()

print("Syntheiszed images are saved in", out_dir)

#### Diffusion model test

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import matplotlib.pyplot as plt

pipe_inpainting = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to(device)

pipe_inpainting.enable_xformers_memory_efficient_attention()

collage_img = Image.open(f"{path_dir}/prepared_input/target_collage.jpg").resize((512, 512))
mask_img = Image.open(f"{path_dir}/prepared_input/mask.png").resize((512, 512))

prompt = """Texture of grey concrete herringbone pavers, rectangular cement tiles in a zigzag pattern,
            weathered surface, white water stains, grunge details, dark grout lines, industrial flooring,
            rough texture, high detail, realistic, top down view."""

image_standard = pipe_inpainting(
    prompt=prompt,
    image=collage_img,
    mask_image=mask_img
).images[0]

image_standard.save(f"{out_dir}/result_diffusion.jpg")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Metodo 1: Standard Inpainting")
plt.imshow(image_standard)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Metodo 3: Self-Rectification (Tuo Risultato)")
plt.imshow(Image.open(f"{out_dir}/result_02.jpg"))
plt.axis('off')
plt.show()

gc.collect()

torch.cuda.empty_cache()